<a href="https://colab.research.google.com/github/LakshmiAiswarya/Hybrid-Movie-Recommendation-System/blob/main/Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
# Check the first few rows
print(movies.head())

   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   
2    206647                                   Spectre   
3     49026                     The Dark Knight Rises   
4     49529                               John Carter   

                                            overview  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   
2  A cryptic message from Bond’s past sends him o...   
3  Following the death of District Attorney Harve...   
4  John Carter is a war-weary, former military ca...   

                                          genres  \
0  [Action, Adventure, Fantasy, Science Fiction]   
1                   [Adventure, Fantasy, Action]   
2                     [Action, Adventure, Crime]   
3               [Action, Crime, Drama, Thriller]   
4           [Action, Adventure, Science Fiction]   

       

In [ ]:
# Fill missing values
movies['overview'] = movies['overview'].fillna('')

# Combine text features
movies['content'] = movies['genres'].apply(lambda x: " ".join(x)) + " " + movies['overview']

In [ ]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Fit and transform the content column
tfidf_matrix = tfidf.fit_transform(movies['content'])

# Check the shape
print(tfidf_matrix.shape)


(4806, 20978)


In [ ]:
# Compute cosine similarity between movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a mapping from movie title to index
movie_indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()


In [ ]:
def get_recommendations(title, cosine_sim=cosine_sim, top_n=10):
    idx = movie_indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]  # Skip the movie itself
    movie_indices_list = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices_list]


In [ ]:
def get_diverse_recommendations(title, cosine_sim=cosine_sim, top_n=10):
    idx = movie_indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:50]  # Take top 50 similar movies first

    # Create a diversity score based on unique genres
    recommended = []
    genres_seen = set()

    for i, score in sim_scores:
        movie_genres = set(movies.iloc[i]['genres'])
        # Only include movie if it introduces a new genre
        if len(genres_seen.intersection(movie_genres)) == 0:
            recommended.append(i)
            genres_seen.update(movie_genres)
        if len(recommended) >= top_n:
            break

    return movies['title'].iloc[recommended]

In [ ]:
movie_name = "The Dark Knight"

print("Standard Recommendations:")
print(get_recommendations(movie_name))

print("\nDiverse Recommendations:")
print(get_diverse_recommendations(movie_name))


Standard Recommendations:
3                         The Dark Knight Rises
428                              Batman Returns
3859    Batman: The Dark Knight Returns, Part 2
299                              Batman Forever
119                               Batman Begins
1360                                     Batman
1361                                     Batman
1182                                        JFK
2509                                  Slow Burn
879                         Law Abiding Citizen
Name: title, dtype: object

Diverse Recommendations:
3      The Dark Knight Rises
506          Despicable Me 2
Name: title, dtype: object


More Improved version

In [ ]:
pip install sentence-transformers

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
# Load the datasets from the zip file
movies_df_raw = pd.read_csv("/content/tmdb_5000_movies.csv")
credits_df_raw = pd.read_csv("/content/tmdb_5000_credits.csv")

# Merge them on 'title' to create the combined movies dataframe
movies = movies_df_raw.merge(credits_df_raw, on="title")

# Helper function to convert string representation of list of dicts to a list of names
# This function re-uses the logic from a previously defined 'convert' function.
import ast # Ensure ast is imported for literal_eval
def _convert_json_list_to_names(text):
    if pd.isna(text):
        return []
    try:
        L = []
        for i in ast.literal_eval(text):
            L.append(i['name'])
        return L
    except (ValueError, SyntaxError):
        return [] # Return empty list for malformed strings

# Apply the conversion to 'genres' and 'keywords' columns
movies['genres'] = movies['genres'].apply(_convert_json_list_to_names)
movies['keywords'] = movies['keywords'].apply(_convert_json_list_to_names)

# Helper function for casting, similar to previous 'convert_cast'
def _convert_cast_names(text):
    L = []
    counter = 0
    if pd.isna(text):
        return []
    try:
        for i in ast.literal_eval(text):
            if counter < 3:
                L.append(i['name'])
                counter += 1
            else:
                break
        return L
    except (ValueError, SyntaxError):
        return []
movies['cast'] = movies['cast'].apply(_convert_cast_names)

# Helper function to fetch director, similar to previous 'fetch_director'
def _fetch_director_name(text):
    L = []
    if pd.isna(text):
        return []
    try:
        for i in ast.literal_eval(text):
            if i['job'] == 'Director':
                L.append(i['name'])
        return L
    except (ValueError, SyntaxError):
        return []
movies['crew'] = movies['crew'].apply(_fetch_director_name)

# Fill missing values for 'overview' (which was 'description' in the original problematic line)
movies['overview'] = movies['overview'].fillna('')

# Combine content: 'genres', 'keywords', 'cast', 'crew' (now lists of strings) and 'overview' (string)
# All lists need to be joined into a single string first.
movies['content'] = movies['genres'].apply(lambda x: " ".join(x)) + " " + \
                    movies['keywords'].apply(lambda x: " ".join(x)) + " " + \
                    movies['cast'].apply(lambda x: " ".join(x)) + " " + \
                    movies['crew'].apply(lambda x: " ".join(x)) + " " + \
                    movies['overview']

In [ ]:
# Load pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all movie content into embeddings
embeddings = model.encode(movies['content'], convert_to_tensor=True)

print("Embeddings shape:", embeddings.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: torch.Size([4809, 384])


In [ ]:
cosine_sim = cosine_similarity(embeddings.cpu(), embeddings.cpu())
movie_indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [ ]:
def get_semantic_recommendations(title, top_n=10):
    idx = movie_indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:50]  # Take top 50 first

    recommended = []
    genres_seen = set()

    for i, score in sim_scores:
        movie_genres = set(movies.iloc[i]['genres'])
        # Include movie if it adds a new genre or plot type
        if len(genres_seen.intersection(movie_genres)) < len(movie_genres):
            recommended.append(i)
            genres_seen.update(movie_genres)
        if len(recommended) >= top_n:
            break

    return movies['title'].iloc[recommended]

In [ ]:
movie_name = "The Dark Knight"
watched = ["The Dark Knight", "Batman Begins"]

print("Semantic Recommendations (Diverse + Smart):")
print(get_semantic_recommendations(movie_name))

# print("\nPersonalized Semantic Recommendations:")
# print(get_semantic_recommendations_personalized(movie_name, watched_titles=watched))

Semantic Recommendations (Diverse + Smart):
3                         The Dark Knight Rises
1361                                     Batman
9            Batman v Superman: Dawn of Justice
3859    Batman: The Dark Knight Returns, Part 2
3824                                   Defendor
163                                    Watchmen
196                                    Megamind
1629                       Deliver Us from Evil
Name: title, dtype: object
